# Download oil-forecasting data

Run this notebook to download or refresh the raw data into `Data/raw/`. Nothing else happens here (no modelling, no plots).

| File | Source | Contents |
|---|---|---|
| `Data/raw/yahoo_daily.csv` | Yahoo Finance | Daily closes: WTI and Brent front-month futures, gasoline, heating oil, natural gas, XLE, S&P 500, VIX, US-dollar index, gold, copper |
| `Data/raw/fred_daily.csv` | FRED (EIA and others) | WTI and Brent **spot** prices, 10y/2y Treasury yields, 10y-2y slope, 10y breakeven inflation, OVX |

**Behaviour**
* **Incremental (default):** if a file already exists, only the last `OVERLAP_DAYS` (90) days are re-downloaded and merged in. Newer values win, so revised data is corrected and history is kept.
* **Full refresh:** set `FULL_REFRESH = True` below (or the environment variable `FULL_REFRESH=1`) to re-download everything from `START_DATE`.
* **Headless / scheduled:** `jupyter nbconvert --to notebook --execute download_data.ipynb --output executed_download.ipynb`

*Notes:* FRED publishes the spot prices with roughly a week's delay, so their last date is earlier than the Yahoo data. Yahoo's `CL=F` / `BZ=F` are unadjusted front-month futures series and contain contract-roll jumps; the FRED spot series do not.

In [9]:
import os
from pathlib import Path
import pandas as pd
import yfinance as yf

START_DATE = "1980-01-01"
OVERLAP_DAYS = 90
FULL_REFRESH = True    # True = ignore existing files and re-download everything

RAW = Path.cwd() / "Data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

YF_TICKERS = {"wti_fut": "CL=F", "brent_fut": "BZ=F", "gasoline": "RB=F", "heating_oil": "HO=F",
              "natgas": "NG=F", "xle": "XLE", "sp500": "^GSPC", "vix": "^VIX",
              "dxy": "DX-Y.NYB", "gold": "GC=F", "copper": "HG=F"}
FRED_SERIES = {"wti_spot": "DCOILWTICO", "brent_spot": "DCOILBRENTEU",
               "ust10y": "DGS10", "ust2y": "DGS2", "curve_10y2y": "T10Y2Y",
               "breakeven10y": "T10YIE", "ovx": "OVXCLS"}
print("Saving to:", RAW)

Saving to: /Users/jianyang/Documents/NUS Master Each Sem/Y2S1/QF5210/Project/OilForecastQF/Data/raw


In [10]:
def download_yahoo(start):
    end = (pd.Timestamp.today() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    cols = {}
    for name, ticker in YF_TICKERS.items():
        try:
            x = yf.download(ticker, start=start, end=end, auto_adjust=False, progress=False, threads=False)
            if isinstance(x.columns, pd.MultiIndex):
                x.columns = x.columns.get_level_values(0)
            if x.empty:
                print(f"  WARNING: no data for {name} ({ticker})")
                continue
            s = x["Close"].copy()
            s.index = pd.to_datetime(s.index).tz_localize(None).normalize()
            cols[name] = s
            print(f"  Yahoo {name:<12} {ticker:<9} {len(s):>5} rows")
        except Exception as exc:
            print(f"  WARNING: Yahoo {name} failed: {exc}")
    return pd.DataFrame(cols).sort_index()

def download_fred(start):
    cols = {}
    for name, series_id in FRED_SERIES.items():
        try:
            x = pd.read_csv(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd={start}")
            s = pd.to_numeric(x.iloc[:, 1], errors="coerce")
            s.index = pd.to_datetime(x.iloc[:, 0])
            cols[name] = s
            print(f"  FRED  {name:<12} {series_id:<13} {s.notna().sum():>5} rows")
        except Exception as exc:
            print(f"  WARNING: FRED {name} failed: {exc}")
    return pd.DataFrame(cols).sort_index()

def update_csv(downloader, path, expected_cols):
    """Download (incrementally if the file is complete) and merge into the CSV; newer values win."""
    old, start = None, START_DATE
    if path.exists() and not FULL_REFRESH:
        old = pd.read_csv(path, index_col=0, parse_dates=True)
        if set(expected_cols) <= set(old.columns):          # a missing column needs its full history
            start = (old.index.max() - pd.Timedelta(days=OVERLAP_DAYS)).strftime("%Y-%m-%d")
    print(f"{path.name}: downloading from {start}")
    new = downloader(start)
    out = new.combine_first(old) if old is not None and len(new) else (old if old is not None else new)
    if out is None or out.empty:
        raise RuntimeError(f"Nothing downloaded and no existing file for {path.name}")
    out = out[~out.index.duplicated(keep="last")].sort_index()
    out.index.name = "date"
    out.to_csv(path)
    return out

## Run the download

In [11]:
market = update_csv(download_yahoo, RAW / "yahoo_daily.csv", YF_TICKERS)
fred = update_csv(download_fred, RAW / "fred_daily.csv", FRED_SERIES)

yahoo_daily.csv: downloading from 1980-01-01
  Yahoo wti_fut      CL=F       6558 rows
  Yahoo brent_fut    BZ=F       4776 rows
  Yahoo gasoline     RB=F       6513 rows
  Yahoo heating_oil  HO=F       6552 rows
  Yahoo natgas       NG=F       6555 rows
  Yahoo xle          XLE        6989 rows
  Yahoo sp500        ^GSPC     11786 rows
  Yahoo vix          ^VIX       9260 rows
  Yahoo dxy          DX-Y.NYB  11915 rows
  Yahoo gold         GC=F       6550 rows
  Yahoo copper       HG=F       6554 rows
fred_daily.csv: downloading from 1980-01-01
  FRED  wti_spot     DCOILWTICO     9512 rows
  FRED  brent_spot   DCOILBRENTEU   9097 rows
  FRED  ust10y       DGS10         11690 rows
  FRED  ust2y        DGS2          11690 rows
  FRED  curve_10y2y  T10Y2Y        11691 rows
  FRED  breakeven10y T10YIE         5944 rows
  FRED  ovx          OVXCLS         4884 rows


## Check what was saved

In [12]:
rows = []
for fname, df in [("yahoo_daily.csv", market), ("fred_daily.csv", fred)]:
    for c in df.columns:
        s = df[c]
        rows.append(dict(file=fname, series=c, first=s.first_valid_index().date(), last=s.last_valid_index().date(), n_obs=int(s.notna().sum())))
summary = pd.DataFrame(rows)
display(summary)
stale = summary[pd.to_datetime(summary["last"]) < pd.to_datetime(summary["last"]).max() - pd.Timedelta(days=3)]
if len(stale):
    print("Series that lag the freshest one by more than 3 days (normal for FRED spot / yields):", ", ".join(stale["series"]))

,file,series,first,last,n_obs
0,yahoo_daily.csv,wti_fut,2000-08-23,2026-10-06,6558
1,yahoo_daily.csv,brent_fut,2007-07-30,2026-10-06,4776
2,yahoo_daily.csv,gasoline,2000-11-01,2026-10-06,6513
3,yahoo_daily.csv,heating_oil,2000-09-01,2026-10-06,6552
4,yahoo_daily.csv,natgas,2000-08-30,2026-10-06,6555
5,yahoo_daily.csv,xle,1998-12-22,2026-10-06,6989
6,yahoo_daily.csv,sp500,1980-01-02,2026-10-06,11786
7,yahoo_daily.csv,vix,1990-01-02,2026-10-06,9260
8,yahoo_daily.csv,dxy,1980-01-02,2026-10-06,11915
9,yahoo_daily.csv,gold,2000-08-30,2026-10-06,6550


Series that lag the freshest one by more than 3 days (normal for FRED spot / yields): wti_spot, brent_spot, ust10y, ust2y
